## Pipeline elements - workbook

### Essentials

In [4]:
# Imports
import psycopg2
from psycopg2 import sql
from dotenv import load_dotenv
import os
import logging
import yaml
import click
import gzip, csv
from contextlib import closing

In [5]:
# Enable logging
logging.basicConfig(
    format="{asctime} - {levelname} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M",
    filename="logs_ingestion_pipeline.log"
)
#logger = logging.getLogger(__name__)
logging.error("Something went wrong")

In [62]:
# Check for existence of credentials, if not found, fail early
load_dotenv(override=True)

def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise EnvironmentError(f"Umgebungsvariable {key} fehlt")
    return val

In [ ]:
# Database credentials
host = require_env("DB_HOST")
dbname = require_env("DB_NAME")
user = require_env("DB_USER")
password = require_env("DB_PASSWORD")

DSN = f"host={host} dbname={dbname} user={user} password={password}"

In [8]:
# Read metadata: tables and column for creating datawarehouse
def read_data_contract(file_path="data_ingestion_contract.yaml") -> list:
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            yaml_data = yaml.safe_load(f)

        if isinstance(yaml_data, dict):
            yaml_data = [yaml_data]
        elif not isinstance(yaml_data, list):
            raise ValueError("YAML content must be a list or dict")
        
        logging.info("Data contract loaded successfully: %s", file_path)
        return yaml_data
    except Exception as e:
        logging.exception("Failed to load data contract: %s", file_path)
        raise


In [9]:
data_contract = read_data_contract()
print(data_contract)

[{'listings': ['estimated_occupancy_l365d', 'number_of_reviews_ly', 'hosts_time_as_user_months', 'property_type', 'host_name', 'listing_url', 'room_type', 'number_of_reviews', 'calendar_last_scraped', 'availability_60', 'host_since', 'accommodates', 'first_review', 'maximum_nights', 'amenities', 'description', 'neighborhood_overview', 'bathrooms', 'last_review', 'review_scores_cleanliness', 'availability_365', 'minimum_nights', 'review_scores_rating', 'latitude', 'calculated_host_listings_count_shared_rooms', 'maximum_minimum_nights', 'availability_90', 'availability_30', 'price', 'number_of_reviews_l30d', 'host_profile_url', 'bathrooms_text', 'minimum_nights_avg_ntm', 'host_thumbnail_url', 'has_availability', 'name', 'price_quote_checkin_date', 'review_scores_communication', 'host_location', 'calculated_host_listings_count', 'host_listings_count', 'host_identity_verified', 'hosts_time_as_host_months', 'source', 'host_id', 'neighbourhood', 'calculated_host_listings_count_private_rooms'

In [10]:
# Create tables and a schema for the raw data
def create_raw_schema_tables(DSN=DSN) -> None:
    try:
        with closing(psycopg2.connect(DSN)) as conn:
            with conn:
                with conn.cursor() as cur:
                    # Create schema
                    cur.execute("CREATE SCHEMA IF NOT EXISTS airbnb_raw_data AUTHORIZATION airbnb_berlin")
                    # Create tables
                    for table, columns in data_contract[0].items():
                        expected_columns = columns

                        if "data_cohort" not in expected_columns:
                            expected_columns.insert(0, "data_cohort")

                        column_def = sql.SQL(", ").join(sql.SQL("{} TEXT").format(
                            sql.Identifier(column)
                        ) for column in expected_columns)

                        query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({})"
                                        ).format(sql.Identifier("airbnb_raw_data",table), column_def)
                        cur.execute(query)

                        # Validate table existence
                        cur.execute("""
                        SELECT column_name FROM information_schema.columns
                        WHERE table_schema = %s AND table_name = %s
                        """, ("airbnb_raw_data", table))

                        actual_columns = {row[0] for row in cur.fetchall()}
                        missing = set(expected_columns) - actual_columns
                        if missing: raise ValueError(f"Table {table}: Missing columns {missing}")

                        logging.info("Validated table %s: %d columns",table,len(actual_columns))

                        actual_columns = {row[0] for row in cur.fetchall()}

            logging.info("Raw schema and table creation successful.")
    except Exception:
        logging.exception("Schema creation failed")
        raise


In [11]:
# List existing data for ingestion
def find_existing_data(filepath: str = require_env("RAWDATAPATH"))-> dict:
    candidates = {}
    for dir in os.listdir(filepath):
        if dir.startswith("data_") and os.path.isdir(dir):
            candidates[dir] = []
            for file in os.listdir(filepath + "/" + dir):
                if file.startswith(("listings","reviews")) and file.endswith(".csv.gz"):
                    candidates[dir].append(file)

    return candidates

In [12]:
find_existing_data()

{'data_26.06.26': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_25.09.23': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_25.12.27': ['listings.csv.gz', 'reviews.csv.gz'],
 'data_26.03.28': ['listings.csv.gz', 'reviews.csv.gz']}

In [13]:
# Detect schema drift
def read_header(folders_specified: tuple) -> list[str]:
    existing_data = find_existing_data()
    headers = {}

    for folder in existing_data:
        if folder in folders_specified:

            headers[folder] = {}

            for file in existing_data[folder]:
                with gzip.open(f"{folder}/{file}", "rt", encoding="utf-8", newline="") as f:
                    header = next(csv.reader(f))
                    headers[folder][file] = header

    return headers

def check_contract(folders_specified: tuple) -> None:
    data_contract = read_data_contract()[0]

    unknown = []
    missing = []

    headers_folders_specified = read_header(folders_specified)
    for folder in headers_folders_specified.keys():
            for table_header in headers_folders_specified[folder].keys():
                data_contract_columns = data_contract[table_header.removesuffix(".csv.gz")]
                real_columns = headers_folders_specified[folder][table_header]
                unknown_columns = [item for item in data_contract_columns if item not in real_columns]
                unknown.append(unknown_columns)                
                missing_columns = [item for item in real_columns if item not in data_contract_columns]
                missing.append(missing_columns)
    unknown = [item for item in unknown if item]
    missing = [item for item in missing if item]
    if unknown:
        logging.error("Schema drift detected: unknown columns in table %s: %s", table_header, sorted(unknown))
        raise ValueError(f"{table_header}: Unknown columns: {sorted(unknown)}")
    if missing:
        logging.warning("Schema drift detected: missing columns in table %s: %s (become NULL)", table_header, sorted(missing))
        raise ValueError(f"{table_header}: Missing columns: {sorted(missing)}")
    return unknown

In [14]:
def check_folders(folders_specified: tuple) -> None:
        existing_folders = find_existing_data().keys()

        invalid_folders = set(folders_specified) - set(existing_folders)

        if invalid_folders: 
                raise click.ClickException(f"Unknown folders: {', '.join(sorted(invalid_folders))}")

In [15]:
# Ingest data from local files into the tables via temporary table
def load_file(folders_for_ingestion: tuple) -> None:
    check_contract(folders_for_ingestion)
    existing_data = find_existing_data()

    headers = read_header(folders_for_ingestion)
    for folder in folders_for_ingestion:
        data_cohort = folder.removeprefix("data_")
        for file in existing_data[folder]:
            table = file.removesuffix(".csv.gz")
            header = headers[folder][file]
            column_definitions = sql.SQL(", ").join(sql.SQL("{} text").format(sql.Identifier(c)) for c in header)
            columns = sql.SQL(", ").join(map(sql.Identifier, header))
            logging.info("Starting ingestion %s/%s",table,file)
            try:
                with closing(psycopg2.connect(DSN)) as conn:
                    with conn:
                        with conn.cursor() as cur:
                            # Temporary staging table
                            cur.execute(sql.SQL("CREATE TEMP TABLE tmp_load ({}) ON COMMIT DROP").format(column_definitions))
                            # Copy csv into staging
                            with gzip.open(f"{folder}/{file}", "rt", encoding="utf-8", newline="") as f:
                                cur.copy_expert("COPY tmp_load FROM STDIN WITH (FORMAT csv, HEADER true)", f)
                            # Validate row count
                            cur.execute("SELECT COUNT(*) FROM tmp_load")
                            staged_rows = cur.fetchone()[0]
                            if staged_rows == 0:
                                raise ValueError(f"No data found in {file}")
                            # Delete existing cohort
                            cur.execute(sql.SQL(
                                "DELETE FROM {} WHERE data_cohort = %s").format(sql.Identifier("airbnb_raw_data", table)), (data_cohort,)
                            )
                            # Insert new cohort into raw table
                            cur.execute(sql.SQL(
                                    "INSERT INTO airbnb_raw_data.{t} (data_cohort, {c}) SELECT %s, {c} FROM tmp_load"
                            ).format(t=sql.Identifier(table), c=columns), (data_cohort,))
                            inserted_rows = cur.rowcount
                logging.info(
                    "Ingestion successful: table=%s, cohort=%s, staged=%d, inserted=%d",
                    table, data_cohort, staged_rows, inserted_rows
                )
            except Exception:
                logging.exception("Ingestion failed: table=%s, file=%s", table, file)
                raise

In [16]:
def run_pipeline(folders_for_ingestion):
    create_raw_schema_tables()
    check_contract(folders_for_ingestion)
    load_file(folders_for_ingestion)

In [17]:
run_pipeline(('data_26.06.26',))

KeyboardInterrupt: 

In [18]:
with closing(psycopg2.connect(DSN)) as conn:
    if conn:
        conn.close()

### Airflow integration

In [85]:
# Imports
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.bash import BashOperator
import ingestion_pipeline
from datetime import datetime, timedelta
import logging
import os
from dotenv import load_dotenv

In [ ]:
# Check for existence of credentials, if not found, fail early
load_dotenv(override=True)

def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise EnvironmentError(f"Umgebungsvariable {key} fehlt")
    return val

In [ ]:
with DAG(
    dag_id="airbnb_berlin_data_ingestion_pipeline",
    start_date=datetime(2026, 10, 1),
    schedule="@weekly",
    catchup=False,
    default_args={"retries": 2, "retry_delay": timedelta(minutes=30)},
) as dag:
    ingestion_pipeline_task = BashOperator(
        task_id="ingestion_pipeline",
        bash_command="/opt/airflow/app-venv/bin/python ingestion_pipeline.py",
        cwd="/opt/airflow/project",
    )

    ingestion_pipeline_task

### New data ingesetion from online

In [39]:
# Imports
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import re
from pathlib import Path
from urllib.request import urlretrieve

In [ ]:
url = "https://insideairbnb.com/en/get-the-data/"

async def get_existing_online_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(url, wait_until="networkidle")
        number_of_h4_elements_before_click = await page.locator("h4").count()

        await page.locator(f"a.showArchivedData[data-city='Berlin']").click()
        await page.locator("h4").nth(number_of_h4_elements_before_click).wait_for()

        soup = BeautifulSoup(await page.content(), "html.parser")
        await browser.close()

        return soup

soup = await get_existing_online_data() # when executed in python script instead of .ipynb-notebook use: asyncio.run(get_existing_online_data())



In [24]:
def extract_city_files(soup):
    start = soup.find(lambda t: t.name == "h3" and t.get_text(strip=True).startswith("Berlin,"))
    files = []

    for element in start.find_all_next(["h3", "a"]):
        if element.name == "h3":
            break
        if "data.insideairbnb.com" in element.get("href", ""):
            files.append({"file": element.get_text(strip=True), "url": element["href"], "date": element["href"].split("/")[-3],})

    return files

In [25]:
berlin_files = extract_city_files(soup)

In [37]:
def candidates_for_download(berlin_files=berlin_files):
    remaining = {}
    raw_candidates = [file for file in berlin_files if file['file'].endswith("listings.csv.gz") or file['file'].endswith("reviews.csv.gz")]
    online_data = {"data_" + re.sub(r'/data/.*','',file['url'].removeprefix("https://data.insideairbnb.com/germany/be/berlin/20").replace("-",".")): {} for file in raw_candidates}

    for file in raw_candidates:
        file_cohort = file['url'].removeprefix("https://data.insideairbnb.com/germany/be/berlin/20").replace("-",".")
        file_cohort = "data_" + re.sub(r'/data/.*','',file_cohort)
        
        file_name = file['file']
        online_data[file_cohort][file_name] = file['url']

    local_data = find_existing_data()

    # cohort wise comparison
    diff_cohorts = [element for element in online_data.keys() if element not in local_data.keys()]
    if diff_cohorts:
        for cohort in diff_cohorts:
            remaining[cohort] = online_data[cohort]

    # table wise comparison
    for cohort in local_data:
        if cohort in online_data.keys():
            diff = [table for table in online_data[cohort].keys() if table not in local_data[cohort]]
            if diff:
                table_diff = {table: online_data[cohort][table] for table in diff if table in online_data[cohort].keys()}
                remaining[cohort] = table_diff

    return remaining
    

In [46]:
for candidate in candidates_for_download():
    for file in candidates_for_download()[candidate].values():
        print(file)

https://data.insideairbnb.com/germany/be/berlin/2026-09-19/data/listings.csv.gz
https://data.insideairbnb.com/germany/be/berlin/2026-09-19/data/reviews.csv.gz


In [69]:
def download_files():
    to_be_downloaded = candidates_for_download()

    data_directory = Path(require_env("DOWNLOAD_PATH"))

    for cohort in to_be_downloaded:
        cohort_directory = data_directory / cohort
        cohort_directory.mkdir(parents=True, exist_ok=True)
        
        for file_name, url in to_be_downloaded[cohort].items():
            urlretrieve(url, cohort_directory / file_name)

In [ ]:
download_files()